In [1]:
!git clone https://github.com/Thinh59/ECAAL.git
%cd ECAAL

Cloning into 'ECAAL'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 50 (delta 13), reused 49 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 957.02 KiB | 7.78 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/kaggle/working/ECAAL


# Fix ASL Bug + Re-run Exp B & C

## 🔴 BUG FOUND:

**ASL Implementation had CRITICAL bug in probability shifting:**

### Paper says (CORRECT):
- Negative branch: `p_shifted = max(p - m, 0)`
- When p < m: p_shifted = 0 → loss_neg = 0 (zero-out easy negatives)

### Old Code (WRONG):
```python
xs_neg_shifted = (xs_neg + self.clip).clamp(max=1.0)  # ← WRONG!
# This is (1-p) + m = (1.05 - p)
# When p=1.0: xs_neg_shifted = 0.05 → log(0.05) = -3.0 (VERY NEGATIVE) ✗
# When p=0.0: xs_neg_shifted = 1.0 → log(1.0) = 0 (zero-out) ✓
# INVERTED! Down-weights easy, up-weights hard!
```

### New Code (CORRECT):
```python
xs_pos_shifted = (xs_pos - self.clip).clamp(min=0)  # max(p - m, 0)
xs_neg_shifted = 1.0 - xs_pos_shifted  # 1 - max(p - m, 0)
# When p=1.0: xs_pos_shifted = 0.95 → xs_neg_shifted = 0.05 → log(0.05) = -3.0 ✓
# When p=0.0: xs_pos_shifted = 0 → xs_neg_shifted = 1.0 → log(1.0) = 0 ✓
# CORRECT! Down-weights easy (p close to 0), keeps hard (p close to 1)!
```

---

## Expected Improvements:
- **Exp B**: Should go from mAP 0.0645 → ~0.65-0.68 (fixing ASL)
- **Exp C**: Should go from mAP 0.1081 → ~0.68-0.70 (fixing ASL + EfficientNet + CBAM)


In [2]:
import os
import sys
from pathlib import Path

# Setup paths
PROJECT_ROOT = Path('/kaggle/working')  # Adjust for local
SRC_DIR = PROJECT_ROOT / 'src'
CONFIG_DIR = PROJECT_ROOT / 'configs'

sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Config dir: {CONFIG_DIR}")
print(f"Configs available: {list(CONFIG_DIR.glob('*.yaml'))}")

Project root: /kaggle/working
Config dir: /kaggle/working/configs
Configs available: []


In [3]:
# Verify the fix was applied
from ECAAL.src.losses import AsymmetricLoss
import torch
import inspect

print("=" * 70)
print("ASL FIX VERIFICATION")
print("=" * 70)

# Check source code
source = inspect.getsource(AsymmetricLoss.forward)
if 'xs_pos_shifted = (xs_pos - self.clip).clamp(min=0)' in source:
    print("✅ FIX APPLIED: Using correct xs_pos_shifted = (xs_pos - self.clip).clamp(min=0)")
    print("   This matches ASL paper: p_shifted = max(p - m, 0)")
else:
    print("❌ OLD BUG STILL PRESENT: xs_neg_shifted = (xs_neg + self.clip)")
    print("   This is INVERTED probability shifting!")

print("\nTesting ASL forward pass...")
asl = AsymmetricLoss(gamma_pos=0, gamma_neg=4, clip=0.05)
logits = torch.randn(4, 10)
targets = torch.randint(0, 2, (4, 10)).float()
loss = asl(logits, targets)
print(f"Loss value: {loss.item():.4f}")
if loss.item() > 0:
    print("✅ Loss is positive (after negation), as expected")
else:
    print("❌ Loss is negative, something is wrong")

ASL FIX VERIFICATION
✅ FIX APPLIED: Using correct xs_pos_shifted = (xs_pos - self.clip).clamp(min=0)
   This matches ASL paper: p_shifted = max(p - m, 0)

Testing ASL forward pass...
Loss value: 5.5365
✅ Loss is positive (after negation), as expected


## Run Exp B: ResNet50 + Fixed ASL

In [4]:
# Import training code
from ECAAL.src.train import run

print("\n" + "="*70)
print("RUNNING EXP B: ResNet50 + Fixed ASL")
print("="*70)

config_b = str(CONFIG_DIR / '/kaggle/working/ECAAL/configs/exp_B_resnet_asl.yaml')
print(f"Config: {config_b}")

best_map_b = run(config_b)
print(f"\n[Result] Exp B Final mAP: {best_map_b:.4f}")


RUNNING EXP B: ResNet50 + Fixed ASL
Config: /kaggle/working/ECAAL/configs/exp_B_resnet_asl.yaml

[Exp] exp_B_resnet_asl | device=cuda
[COCO 2017 train] 117,266 images loaded
[COCO 2017 val] 4,952 images loaded


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

[Model] resnet50 | CBAM=False | Params=23.67M | FeatChannels=2048
[Loss] ASL | params: {'gamma_pos': 0, 'gamma_neg': 4, 'clip': 0.05}

--- Epoch 1/20 ---
  [50/1832] loss=3.8380
  [100/1832] loss=3.6930
  [150/1832] loss=3.5707
  [200/1832] loss=3.4770
  [250/1832] loss=3.3856
  [300/1832] loss=3.3005
  [350/1832] loss=3.2260
  [400/1832] loss=3.1609
  [450/1832] loss=3.1091
  [500/1832] loss=3.0604
  [550/1832] loss=3.0158
  [600/1832] loss=2.9766
  [650/1832] loss=2.9324
  [700/1832] loss=2.8907
  [750/1832] loss=2.8462
  [800/1832] loss=2.8049
  [850/1832] loss=2.7604
  [900/1832] loss=2.7197
  [950/1832] loss=2.6825
  [1000/1832] loss=2.6448
  [1050/1832] loss=2.6086
  [1100/1832] loss=2.5739
  [1150/1832] loss=2.5408
  [1200/1832] loss=2.5092
  [1250/1832] loss=2.4765
  [1300/1832] loss=2.4463
  [1350/1832] loss=2.4172
  [1400/1832] loss=2.3881
  [1450/1832] loss=2.3612
  [1500/1832] loss=2.3355
  [1550/1832] loss=2.3108
  [1600/1832] loss=2.2868
  [1650/1832] loss=2.2645
  [1700/

## Run Exp C: EfficientNet-B0 + CBAM + Fixed ASL

In [5]:
print("\n" + "="*70)
print("RUNNING EXP C: EfficientNet-B0 + CBAM + Fixed ASL")
print("="*70)

config_c = str(CONFIG_DIR / '/kaggle/working/ECAAL/configs/exp_C_efficientnet_cbam_asl.yaml')
print(f"Config: {config_c}")

best_map_c = run(config_c)
print(f"\n[Result] Exp C Final mAP: {best_map_c:.4f}")


RUNNING EXP C: EfficientNet-B0 + CBAM + Fixed ASL
Config: /kaggle/working/ECAAL/configs/exp_C_efficientnet_cbam_asl.yaml

[Exp] exp_C_efficientnet_cbam_asl | device=cuda
[COCO 2017 train] 117,266 images loaded
[COCO 2017 val] 4,952 images loaded


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


[Model] efficientnet_b0 | CBAM=True | Params=3.63M | FeatChannels=320
[Loss] ASL | params: {'gamma_pos': 0, 'gamma_neg': 4, 'clip': 0.05}

--- Epoch 1/20 ---


/kaggle/working/ECAAL/src/train.py:46: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  [50/1832] loss=4.1697
  [100/1832] loss=4.1181
  [150/1832] loss=4.0571
  [200/1832] loss=3.9995
  [250/1832] loss=3.9363
  [300/1832] loss=3.8558
  [350/1832] loss=3.7650
  [400/1832] loss=3.6728
  [450/1832] loss=3.5815
  [500/1832] loss=3.4899
  [550/1832] loss=3.4076
  [600/1832] loss=3.3252
  [650/1832] loss=3.2484
  [700/1832] loss=3.1749
  [750/1832] loss=3.1086
  [800/1832] loss=3.0476
  [850/1832] loss=2.9912
  [900/1832] loss=2.9373
  [950/1832] loss=2.8886
  [1000/1832] loss=2.8435
  [1050/1832] loss=2.7989
  [1100/1832] loss=2.7583
  [1150/1832] loss=2.7187
  [1200/1832] loss=2.6822
  [1250/1832] loss=2.6472
  [1300/1832] loss=2.6153
  [1350/1832] loss=2.5827
  [1400/1832] loss=2.5516
  [1450/1832] loss=2.5241
  [1500/1832] loss=2.4968
  [1550/1832] loss=2.4706
  [1600/1832] loss=2.4457
  [1650/1832] loss=2.4232
  [1700/1832] loss=2.4025
  [1750/1832] loss=2.3810
  [1800/1832] loss=2.3604
  train_loss=2.3473

--- Epoch 2/20 ---
  [50/1832] loss=1.5938
  [100/1832] loss=1.

## Compare Results

In [6]:
import pandas as pd

# Results after fix
results_comparison = pd.DataFrame({
    'Experiment': ['Exp A (BCE)', 'Exp B (ASL - OLD BUG)', 'Exp B (ASL - FIXED)', 'Exp C (ASL - OLD BUG)', 'Exp C (ASL - FIXED)', 'Exp D (Focal)'],
    'mAP': [0.6840, 0.0645, f"{best_map_b:.4f}", 0.1081, f"{best_map_c:.4f}", 0.6911],
    'Status': ['✅ Good', '❌ Bug', '✅ Fixed', '❌ Bug', '✅ Fixed', '✅ Good']
})

print("\n" + "="*80)
print("RESULTS COMPARISON: Before & After ASL Bug Fix")
print("="*80)
print(results_comparison.to_string(index=False))
print("\n" + "="*80)

print(f"\n📊 ANALYSIS:")
print(f"  • Exp B improvement: {0.0645:.4f} → {best_map_b:.4f} (+{(best_map_b - 0.0645)*100:.1f}%)")
print(f"  • Exp C improvement: {0.1081:.4f} → {best_map_c:.4f} (+{(best_map_c - 0.1081)*100:.1f}%)")
print(f"\n  • After fix, Exp B vs Exp D: {best_map_b:.4f} vs 0.6911 ({(best_map_b/0.6911)*100:.1f}% of Exp D)")
print(f"  • After fix, Exp C vs Exp D: {best_map_c:.4f} vs 0.6911 ({(best_map_c/0.6911)*100:.1f}% of Exp D)")


RESULTS COMPARISON: Before & After ASL Bug Fix
           Experiment     mAP  Status
          Exp A (BCE)   0.684  ✅ Good
Exp B (ASL - OLD BUG)  0.0645   ❌ Bug
  Exp B (ASL - FIXED)  0.7913 ✅ Fixed
Exp C (ASL - OLD BUG)  0.1081   ❌ Bug
  Exp C (ASL - FIXED)  0.7516 ✅ Fixed
        Exp D (Focal)  0.6911  ✅ Good


📊 ANALYSIS:
  • Exp B improvement: 0.0645 → 0.7913 (+72.7%)
  • Exp C improvement: 0.1081 → 0.7516 (+64.4%)

  • After fix, Exp B vs Exp D: 0.7913 vs 0.6911 (114.5% of Exp D)
  • After fix, Exp C vs Exp D: 0.7516 vs 0.6911 (108.8% of Exp D)


## Save Results to JSON

In [7]:
import json
from datetime import datetime

results_fix = {
    'timestamp': datetime.now().isoformat(),
    'bug_fixed': 'ASL probability shifting (xs_neg_shifted bug)',
    'fix_description': 'Changed xs_neg_shifted = (xs_neg + clip) to xs_pos_shifted = (xs_pos - clip).clamp(min=0)',
    'results': {
        'exp_b': {
            'before_fix': 0.0645,
            'after_fix': float(best_map_b),
            'improvement_pct': (best_map_b - 0.0645) / 0.0645 * 100
        },
        'exp_c': {
            'before_fix': 0.1081,
            'after_fix': float(best_map_c),
            'improvement_pct': (best_map_c - 0.1081) / 0.1081 * 100
        },
        'exp_a': 0.6840,  # baseline for reference
        'exp_d': 0.6911,  # baseline for reference
    }
}

output_file = PROJECT_ROOT / 'asl_bugfix_results.json'
with open(output_file, 'w') as f:
    json.dump(results_fix, f, indent=2)

print(f"\n✅ Results saved to: {output_file}")
print(json.dumps(results_fix, indent=2))


✅ Results saved to: /kaggle/working/asl_bugfix_results.json
{
  "timestamp": "2026-04-20T10:36:07.729020",
  "bug_fixed": "ASL probability shifting (xs_neg_shifted bug)",
  "fix_description": "Changed xs_neg_shifted = (xs_neg + clip) to xs_pos_shifted = (xs_pos - clip).clamp(min=0)",
  "results": {
    "exp_b": {
      "before_fix": 0.0645,
      "after_fix": 0.7912570116919531,
      "improvement_pct": 1126.755056886749
    },
    "exp_c": {
      "before_fix": 0.1081,
      "after_fix": 0.7516180673175815,
      "improvement_pct": 595.2988596832392
    },
    "exp_a": 0.684,
    "exp_d": 0.6911
  }
}
